# Pano2Room from soda-can reflections

1. Select **Runtime → Change runtime type → GPU**. Choose an NVIDIA T4, L4, or A100 if available.
2. Run the cells in order. The first section mounts Drive and prompts for the exported `input_panorama.png` if it isn't already in the project folder on Drive.
3. Look for `3DGS.ply` and `GS_render_video.mp4` in the printed Drive run folder.

The first setup compiles CUDA extensions and downloads model weights. Keep the runtime connected. This notebook runs the [authors' Pano2Room code](https://github.com/TrickyGo/Pano2Room) with compatibility fixes for the CUDA rasterizer and current model download location. The completed 3D scene includes inferred depth and synthesized appearance.

Original work: [Pu, Zhao, and Lian, Pano2Room, SIGGRAPH Asia 2024](https://doi.org/10.1145/3680528.3687616).

## 1. Select the panorama

In [ ]:
from pathlib import Path
import platform
import shutil
import subprocess

if platform.system() != "Linux" or shutil.which("nvidia-smi") is None:
    raise RuntimeError("Switch to a GPU runtime in Colab and rerun this cell.")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv"], check=True)

from google.colab import drive, files
from IPython.display import display
from PIL import Image

drive.mount("/content/drive")
DRIVE_PROJECT = Path("/content/drive/MyDrive/cs180_final_project")
PANORAMA_ON_DRIVE = DRIVE_PROJECT / "results/pano2room/input_panorama.png"
RESULTS_ON_DRIVE = DRIVE_PROJECT / "pano2room_runs"
CHECKPOINTS_ON_DRIVE = DRIVE_PROJECT / "pano2room_checkpoints"

work = Path("/content/can_pano2room")
work.mkdir(exist_ok=True)
panorama = work / "input_panorama.png"

if PANORAMA_ON_DRIVE.is_file():
    source = PANORAMA_ON_DRIVE
else:
    print("Upload the camera-oriented input_panorama.png exported by our project.")
    uploads = files.upload()
    if len(uploads) != 1:
        raise ValueError("Choose one exported panorama.")
    source = Path(next(iter(uploads)))

with Image.open(source) as image:
    if image.width != 2 * image.height:
        raise ValueError("Expected an equirectangular image with width twice its height.")
    display(image.copy())
    image.convert("RGB").save(panorama)
print(f"Outputs will be saved under {RESULTS_ON_DRIVE}")

## 2. Build the CUDA environment

The notebook uses Python 3.10, CUDA 11.8, and PyTorch 2.4.1 on Colab's local disk. The CUDA compiler is installed separately from the Python environment. It installs the [official prebuilt PyTorch3D package](https://conda.anaconda.org/pytorch3d/linux-64) and compiles the other CUDA extensions. The package installs can be rerun within the same session. Start from a fresh runtime if changing the CUDA or PyTorch versions.

In [ ]:
import os

repo = work / "Pano2Room"
commit = "bbf93ae57086ed700edc6ee445852d4457a9d704"
if not (repo / ".git").is_dir():
    subprocess.run(["git", "init", str(repo)], check=True)
if not (repo / "pano2room.py").exists():
    subprocess.run(
        ["git", "fetch", "--depth", "1", "https://github.com/TrickyGo/Pano2Room.git", commit],
        cwd=repo, check=True,
    )
    subprocess.run(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=repo, check=True)

mamba = work / "bin/micromamba"
if not mamba.is_file():
    archive = work / "micromamba.tar.bz2"
    subprocess.run(["curl", "-fsSL", "--retry", "3",
                    "https://micro.mamba.pm/api/micromamba/linux-64/latest",
                    "-o", str(archive)], check=True)
    subprocess.run(["tar", "-xjf", str(archive), "-C", str(work), "bin/micromamba"], check=True)

prefix = work / "env"
mamba_env = {**os.environ, "MAMBA_ROOT_PREFIX": str(work / "mamba")}
if not (prefix / "bin/python").exists():
    subprocess.run(
        [str(mamba), "create", "-y", "-p", str(prefix),
         "-c", "conda-forge",
         "python=3.10", "pip=24.0", "gcc_linux-64=11",
         "gxx_linux-64=11", "ninja"],
        env=mamba_env, check=True,
    )

def in_env(*command, **options):
    return subprocess.run(
        [str(mamba), "run", "-p", str(prefix), *command],
        env=mamba_env, cwd=repo, check=True, **options,
    )

in_env("python", "-m", "pip", "install", "--index-url", "https://download.pytorch.org/whl/cu118",
       "torch==2.4.1", "torchvision==0.19.1")

In [ ]:
packages = """
numpy==1.26.4 scipy==1.13.1 opencv-python-headless==4.10.0.84
pillow==10.2.0 timm==0.6.13 diffusers==0.30.0 transformers==4.44.2
huggingface-hub==0.24.7 accelerate==0.33.0 peft==0.12.0
pytorch-lightning==1.9.0 torchmetrics==1.3.2
albumentations==0.5.2 imgaug==0.4.0 scikit-image==0.21.0 scikit-learn==1.3.1
kornia==0.7.0 moviepy==1.0.3 imageio-ffmpeg matplotlib==3.7.3
open3d==0.18.0 trimesh==3.23.5 plyfile==1.0.3
omegaconf h5py icecream easydict tqdm yacs iopath fvcore einops
lmdb webdataset lpips tensorboard rich gdown addict packaging wheel setuptools<80
pyquaternion pymeshlab imWatermark easydict roma torch-efficient-distloss
"""
in_env("python", "-m", "pip", "install", *dict.fromkeys(packages.split()))

gpu_arch = in_env(
    "python", "-c",
    "import torch; print('.'.join(map(str, torch.cuda.get_device_capability())))",
    capture_output=True, text=True,
).stdout.strip()
if tuple(map(int, gpu_arch.split("."))) > (9, 0):
    raise RuntimeError("This CUDA 11.8 build targets T4, L4, A100, or H100. Select one of those GPUs.")

# Keep this toolkit separate so Colab's system CUDA or the Python env cannot override it.
cuda_home = work / "cuda-11.8"
nvcc = cuda_home / "bin/nvcc"
if not nvcc.is_file():
    subprocess.run(
        [str(mamba), "create", "-y", "-p", str(cuda_home),
         "--override-channels", "--strict-channel-priority",
         "-c", "nvidia/label/cuda-11.8.0", "-c", "conda-forge",
         "cuda-nvcc=11.8.89", "cuda-cudart-dev=11.8.89",
         "cuda-cccl=11.8.89", "cuda-libraries-dev=11.8.0",
         "cuda-driver-dev=11.8.89"],
        env=mamba_env, check=True,
    )

# tiny-cuda-nn links against the CUDA driver stub during compilation.
if not list((cuda_home / "conda-meta").glob("cuda-driver-dev-11.8.89-*.json")):
    subprocess.run(
        [str(mamba), "install", "-y", "-p", str(cuda_home),
         "--override-channels", "--strict-channel-priority",
         "-c", "nvidia/label/cuda-11.8.0", "-c", "conda-forge",
         "cuda-driver-dev=11.8.89"],
        env=mamba_env, check=True,
    )
driver_stub = next(cuda_home.glob("**/libcuda.so"), None)
if driver_stub is None:
    raise FileNotFoundError(f"CUDA driver link stub was not found under {cuda_home}.")

# Pass overrides after Micromamba activates the Python environment.
build_env = [
    f"CUDA_HOME={cuda_home}", f"CUDA_PATH={cuda_home}", f"PYTORCH_NVCC={nvcc}",
    f"PATH={cuda_home / 'bin'}:{prefix / 'bin'}:{os.environ['PATH']}",
    f"LIBRARY_PATH={driver_stub.parent}",
    f"CC={prefix / 'bin/x86_64-conda-linux-gnu-gcc'}",
    f"CXX={prefix / 'bin/x86_64-conda-linux-gnu-g++'}",
    f"TORCH_CUDA_ARCH_LIST={gpu_arch}", "FORCE_CUDA=1", "MAX_JOBS=2",
]
in_env("env", *build_env, "python", "-c", """
import re
import subprocess
import torch
from pathlib import Path
from torch.utils.cpp_extension import CUDA_HOME

nvcc = Path(CUDA_HOME) / "bin/nvcc"
version = subprocess.check_output([str(nvcc), "--version"], text=True)
match = re.search(r"release ([0-9]+[.][0-9]+)", version)
if not match or match.group(1) != torch.version.cuda:
    raise RuntimeError(f"{nvcc}: {version.strip()}. PyTorch uses CUDA {torch.version.cuda}.")
print(f"CUDA {torch.version.cuda} compiler verified: {nvcc}")
""")

# Install the exact Linux binary for the Python, CUDA, and PyTorch already in this environment.
pytorch3d_check = """
import sys
import torch
import pytorch3d
from pytorch3d.ops import knn_points

assert sys.version_info[:2] == (3, 10), sys.version
assert torch.__version__.split("+")[0] == "2.4.1", torch.__version__
assert torch.version.cuda == "11.8", torch.version.cuda
assert pytorch3d.__version__ == "0.7.8", pytorch3d.__version__
points = torch.zeros((1, 2, 3), device="cuda")
knn_points(points, points, K=1)
torch.cuda.synchronize()
print("PyTorch3D CUDA check passed.")
"""
in_env("python", "-c", """
import sys
import torch
assert sys.version_info[:2] == (3, 10), sys.version
assert torch.__version__.split("+")[0] == "2.4.1", torch.__version__
assert torch.version.cuda == "11.8", torch.version.cuda
""")
build = "pytorch3d-0.7.8-py310_cu118_pyt241"
if not (prefix / "conda-meta" / f"{build}.json").exists():
    spec = work / "pytorch3d-explicit.txt"
    spec.write_text(
        "@EXPLICIT\n"
        f"https://conda.anaconda.org/pytorch3d/linux-64/{build}.tar.bz2\n"
    )
    subprocess.run(
        [str(mamba), "install", "-y", "-p", str(prefix), "--file", str(spec)],
        env=mamba_env, check=True,
    )
in_env("python", "-c", pytorch3d_check)

extensions = [
    ("diff_gaussian_rasterization", "git+https://github.com/JonathonLuiten/diff-gaussian-rasterization-w-depth.git"),
    ("simple_knn", "git+https://github.com/camenduru/simple-knn.git"),
]
# Some predictor modules use the tiny-cuda-nn encoder.
if any("import tinycudann" in file.read_text(errors="replace") for file in (repo / "modules").rglob("*.py")):
    extensions.append(("tinycudann", "git+https://github.com/NVlabs/tiny-cuda-nn.git@v2.0#subdirectory=bindings/torch"))
build_log = work / "extension-build.log"
build_log.write_text("")
for module, url in extensions:
    result = subprocess.run(
        [str(mamba), "run", "-p", str(prefix), "python", "-c", f"import torch; import {module}"],
        cwd=repo, env=mamba_env, capture_output=True, text=True,
    )
    if result.returncode:
        print(f"Building {module}. Full output: {build_log}", flush=True)
        command = [str(mamba), "run", "-p", str(prefix), "env", *build_env,
                   "python", "-m", "pip",
                   "install", "--verbose", "--no-build-isolation", "--no-deps", url]
        with build_log.open("a") as log:
            print(f"\n=== {module} ===", file=log, flush=True)
            build_result = subprocess.run(command, cwd=repo, env=mamba_env, stdout=log,
                                          stderr=subprocess.STDOUT)
        if build_result.returncode:
            from collections import deque

            with build_log.open() as log:
                print("".join(deque(log, maxlen=60)))
            raise RuntimeError(f"{module} build failed. See the output above and {build_log}.")

in_env("python", "-c", """
import torch
from simple_knn._C import distCUDA2
from diff_gaussian_rasterization import GaussianRasterizer
import tinycudann
import timm
assert timm.is_model("vit_base_resnet50_384")
print("CUDA extensions loaded on", torch.cuda.get_device_name(0))
""")

## 3. Prepare Pano2Room

The notebook uses the published depth-aware rasterizer and the [Stable Diffusion 2 inpainting mirror](https://huggingface.co/sd2-community/stable-diffusion-2-inpainting). It downloads the three checkpoints into a reusable Drive folder, then copies them to local disk. If a download is unavailable, use the [authors' checkpoint link](https://github.com/TrickyGo/Pano2Room/tree/main/checkpoints) and put the three named files directly in the printed checkpoint folder, then rerun this cell.

In [ ]:
# Keep these edits inside the downloaded checkout.
rasterizer = repo / "gaussian_renderer/__init__.py"
source_text = rasterizer.read_text()
source_text = source_text.replace("from depth_diff_gaussian_rasterization_min import",
                                  "from diff_gaussian_rasterization import")
source_text = source_text.replace("        debug=opt.debug\n", "")
compile(source_text, str(rasterizer), "exec")
rasterizer.write_text(source_text)

inpainter = repo / "modules/inpainters/SDFT_inpainter.py"
source_text = inpainter.read_text()
source_text = source_text.replace('"stabilityai/stable-diffusion-2-inpainting"',
                                  '"sd2-community/stable-diffusion-2-inpainting"')
source_text = source_text.replace(', variant="fp16"', '')
inpainter.write_text(source_text)

# Initialize geometry from the rendered view even when it has no holes.
pipeline = repo / "pano2room.py"
source_text = pipeline.read_text()
pose_update = """                perf_pose = pose.clone()
                perf_pose[0,3], perf_pose[1,3], perf_pose[2,3] = -pose[0,3], pose[2,3], 0
"""
if pose_update in source_text:
    source_text = source_text.replace(pose_update, "", 1)
    marker = "            if pano_inpaint_mask.min().item() < .5:"
    if source_text.count(marker) != 1:
        raise RuntimeError("The upstream inpainting loop changed.")
    source_text = source_text.replace(
        marker,
        "            perf_pose = pose.clone()\n"
        "            perf_pose[0,3], perf_pose[1,3], perf_pose[2,3] = -pose[0,3], pose[2,3], 0\n"
        "            normals = None\n" + marker,
    )
    compile(source_text, str(pipeline), "exec")
    pipeline.write_text(source_text)

checkpoint_names = (
    "big-lama.ckpt", "omnidata_dpt_depth_v2.ckpt", "omnidata_dpt_normal_v2.ckpt",
)
CHECKPOINTS_ON_DRIVE.mkdir(parents=True, exist_ok=True)

def checkpoint_ready(path):
    return path.is_file() and path.stat().st_size > 1_000_000

if any(not checkpoint_ready(CHECKPOINTS_ON_DRIVE / name) for name in checkpoint_names):
    print(f"Checkpoint cache: {CHECKPOINTS_ON_DRIVE}")
    # These file IDs come from the Pano2Room setup fork's download script.
    download_dir = work / "checkpoint_downloads"
    download_dir.mkdir(exist_ok=True)
    downloader = """
import sys
from pathlib import Path
import gdown

destination = Path(sys.argv[1])
for file_id in sys.argv[2:]:
    try:
        gdown.download(id=file_id, output=str(destination) + "/", quiet=False)
    except Exception as error:
        print(f"Download failed for {file_id}: {error}")
"""
    in_env("python", "-c", downloader, str(download_dir),
           "1H5CHOsm_yAxZI9a5hv9tyZmh5CMjJxap",
           "18S9ycwHi07hzPdLORsAQFFORTeovdo4E",
           "1gMBrl51AZZr6ANy8d77KFXb7oiYzMDjw")
    for name in checkpoint_names:
        downloaded = download_dir / name
        cached = CHECKPOINTS_ON_DRIVE / name
        if checkpoint_ready(downloaded) and not checkpoint_ready(cached):
            shutil.copy2(downloaded, cached)

missing = [name for name in checkpoint_names if not checkpoint_ready(CHECKPOINTS_ON_DRIVE / name)]
if missing:
    raise FileNotFoundError(f"Copy {', '.join(missing)} into {CHECKPOINTS_ON_DRIVE} and rerun this cell.")

for name in checkpoint_names:
    local = repo / "checkpoints" / name
    if not checkpoint_ready(local):
        shutil.copy2(CHECKPOINTS_ON_DRIVE / name, local)

shutil.copy2(panorama, repo / "input/input_panorama.png")
in_env("python", "-c", "import gaussian_renderer; from modules.inpainters import PanoPersFusionInpainter; from modules.geo_predictors import PanoJointPredictor")
print("Panorama, checkpoints, and model imports are ready.")

## 4. Reconstruct the room

Each run gets a new timestamped folder on Drive. The notebook streams the log there and writes the scene and videos there as they finish.

In [ ]:
from datetime import datetime, timezone

run_id = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f")
run_dir = RESULTS_ON_DRIVE / run_id
run_dir.mkdir(parents=True)
shutil.copy2(panorama, run_dir / "input_panorama.png")

output = repo / "output/Pano2Room-results"
output.parent.mkdir(exist_ok=True)
if output.is_symlink():
    output.unlink()
elif output.exists():
    raise FileExistsError(f"Move the previous local output folder first: {output}")
output.symlink_to(run_dir, target_is_directory=True)
print(f"Run folder: {run_dir}", flush=True)

with (run_dir / "run.log").open("w", buffering=1) as log:
    command = [str(mamba), "run", "-p", str(prefix), "python", "-u", "pano2room.py"]
    with subprocess.Popen(command, cwd=repo, env=mamba_env, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, errors="replace") as process:
        for line in process.stdout:
            log.write(line)
            print(line, end="", flush=True)
        if process.wait():
            raise RuntimeError(f"Pano2Room stopped. See {run_dir / 'run.log'}.")

scene = run_dir / "3DGS.ply"
video = run_dir / "GS_render_video.mp4"
for path in (scene, video):
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Expected output is missing or empty: {path}")
print(f"3D scene: {scene}\nWalkthrough: {video}")

## 5. View and download

Preview the walkthrough below. Download `3DGS.ply` from the run folder in Drive and drop it into [SuperSplat](https://superspl.at/editor) to move through the scene.

In [ ]:
from IPython.display import Video, display

display(Video(str(video), embed=True, width=640))